# Stage 6 · Extend the dense Lego fit by 10,000 updates

Attach the original Stage 0 data/code kit and `lego-dense-vanilla-s42-step50000.zip` as private inputs. Expanded inputs work. Select **T4 ×2** and run the cells in order. A fresh session can also attach the latest Stage 6 ZIP to continue. Internet and LPIPS are unnecessary.

This is a separate candidate protocol. The original 50,000 updates used the scored schedule. Starting at update 50,001, keep Adam, both RNG states, 512 rays per update and the final learning rate of 0.00005. The candidate endpoint is 75,000. This notebook stops at **60,000** for a training-only review. The 25 held-out test cameras stay reserved.

Expect roughly 32 minutes for 10,000 updates at 5.285 updates/s and 3 minutes for eight training images, plus setup and export. Save the checkpoint links after each segment. Return **lego-dense-vanilla-s42-stage6-review.zip** from the last cell.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import base64
import hashlib
import json
import os
import shutil
import statistics
import sys
import time
import zipfile

import torch
from IPython.display import HTML, display

inputs = Path("/kaggle/input")
outputs = Path("/kaggle/working")
work = outputs / "recovery-stage6-dense-extension"
work.mkdir(parents=True, exist_ok=True)
if "input_cache" not in globals():
    input_cache = TemporaryDirectory(prefix="nerf-stage6-")
cache = Path(input_cache.name)
device = torch.device("cuda:0")
if not torch.cuda.is_available():
    raise RuntimeError("Select the Kaggle T4 accelerator and restart this notebook.")

parent_id = "08ff8d2fceda43759554aa6470054853"
parent_sha = "a77230d4e17b3769445cf05e26b2bda7f094bf078418ff7d6366d906e19da8be"
data_sha = "bdc22418467720a3dbf5555318532caf17c250a05433f8553ae58372f11e89e2"
code_sha = "4b85a7f9d5836739fb562c5fb3ff25fa68b063073bbea539e38ed1130f088eb5"
relative_run = Path("runs/lego/dense-vanilla/seed-42")
run = work / relative_run
manifest = work / "manifests/lego.json"
manifest.parent.mkdir(parents=True, exist_ok=True)
fork_step, review_step, candidate_end = 50_000, 60_000, 75_000
protocol_id = "50k-exponential-then-25k-hold-v1"

def read(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

def unpack(path):
    # Separate identically named ZIPs in different Kaggle datasets.
    folder = cache / hashlib.sha256(str(path).encode()).hexdigest()
    folder.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if not (folder / name).resolve().is_relative_to(folder.resolve()):
                raise ValueError(f"Unexpected ZIP member {name}")
            archive.extract(name, folder)
    return folder

kits = sorted(inputs.rglob("RECOVERY_STAGE0.txt"))
if kits:
    repo = kits[0].parent
else:
    archives = sorted(inputs.rglob("stage0-input.zip"))
    if not archives:
        raise FileNotFoundError("Attach the original Stage 0 dataset.")
    repo = unpack(archives[0])

def parent_root():
    for marker in sorted(inputs.rglob("stage3-review.json")):
        record = read(marker)
        if (record.get("run_id"), record.get("checkpoint_step"), record.get("checkpoint_sha256")) == (
            parent_id, fork_step, parent_sha):
            return marker.parent
    archives = sorted(inputs.rglob("lego-dense-vanilla-s42-step50000.zip"))
    if not archives:
        raise FileNotFoundError("Attach the reviewed dense step-50,000 ZIP or its expanded output.")
    return unpack(archives[0])

source = parent_root()
print("Selected", torch.cuda.get_device_name(device), "for the continuation.")

In [ ]:
sys.path.insert(0, str(repo))
from recovery.data import verify_manifest
from recovery.media import alpha_mask, psnr, render_maps
from recovery.training import fingerprint, implementation_hash, load_state, make_model, settings, train, write_json
from sparse_nerf.data import SceneDataset

parent_run = source / relative_run
parent_meta = read(parent_run / "run.json")
parent_record = read(source / "stage3-review.json")
frozen = verify_manifest(source / "manifests/lego.json", repo / "data/blender/lego")
write_json(manifest, frozen)
assert frozen["data_hash"] == parent_meta["data_hash"] == data_sha
assert implementation_hash() == parent_meta["training_config"]["implementation_hash"] == code_sha
assert hashlib.sha256((repo / "recovery/media.py").read_text(encoding="utf-8").encode()).hexdigest() == "fd13945accd1ec6938b7ebceccec31fe1f8716dc2a2ad51d84068d560e09a3af"
assert sha(parent_run / "checkpoint.pt") == parent_record["checkpoint_sha256"] == parent_sha
assert parent_record["run_id"] == parent_meta["run_id"] == parent_id
assert parent_record["complete"] and parent_record["checkpoint_step"] == fork_step
assert parent_meta["training_config"] == settings("main", frozen["camera"])
assert (parent_meta["scene"], parent_meta["protocol"], parent_meta["train_views"], parent_meta["seed"]) == (
    "lego", "dense-vanilla", 100, 42)

# The frozen trainer accepts a constant rate when its two LR endpoints are equal.
# Preserve the actual first-phase schedule inside the comparison settings.
prefix_cfg = parent_meta["training_config"]
cfg = dict(prefix_cfg, steps=candidate_end, lr=prefix_cfg["lr_final"], lr_final=prefix_cfg["lr_final"],
           prefix_config=prefix_cfg, prefix_updates=fork_step, schedule=protocol_id)
child_id = hashlib.sha256(f"{parent_sha}/{protocol_id}".encode()).hexdigest()[:32]
lineage = dict(parent_run_id=parent_id, parent_checkpoint_sha256=parent_sha, parent_step=fork_step)
meta = dict(parent_meta, run_id=child_id, training_config=cfg, lineage=lineage)

def checked_state(directory, expected, low, high):
    if read(directory / "run.json") != expected:
        raise ValueError(f"Different run metadata at {directory}")
    state = load_state(directory, expected)
    step = state["step"]
    slots = state["optimizer"]["state"]
    if not low <= step <= high or len(slots) != 48 or any(float(s["step"]) != step for s in slots.values()):
        raise ValueError("Checkpoint step or Adam counters differ from the recorded run")
    tensors = [*state["model"].values(), *(s[k] for s in slots.values() for k in ("exp_avg", "exp_avg_sq"))]
    if not all(t.isfinite().all().item() for t in tensors):
        raise ValueError("Nonfinite model or Adam moments")
    if not all(isinstance(state[k], torch.Tensor) and state[k].numel() for k in ("cpu_rng", "cuda_rng")):
        raise ValueError("Missing checkpoint RNG state")
    return state

def valid_receipt(receipt):
    return (receipt.get("run_id") == child_id and receipt.get("lineage") == lineage and
            receipt.get("schedule") == protocol_id and fork_step <= receipt["checkpoint_step"] <= review_step)

# Find the furthest saved point, including the current writable session.
choices = []
for local_root in [work, *sorted(outputs.glob("recovery-stage6-dense-extension-restored-*"))]:
    local_run = local_root / relative_run
    if local_run.is_dir():
        state = checked_state(local_run, meta, fork_step, review_step)
        choices.append((state["step"], 2, local_run, sha(local_run / "checkpoint.pt")))
        del state
for marker in inputs.rglob("stage6-review.json"):
    receipt = read(marker)
    if valid_receipt(receipt):
        choices.append((receipt["checkpoint_step"], 1, marker.parent / relative_run, receipt["checkpoint_sha256"]))
for name in ("lego-dense-vanilla-s42-extension-*.zip", "lego-dense-vanilla-s42-stage6-review.zip"):
    for path in inputs.rglob(name):
        with zipfile.ZipFile(path) as archive:
            receipt = json.loads(archive.read("stage6-review.json"))
        if valid_receipt(receipt):
            choices.append((receipt["checkpoint_step"], 0, path, receipt["checkpoint_sha256"]))

if choices:
    step, kind, best, digest = max(choices)
    if len({entry[3] for entry in choices if entry[0] == step}) > 1:
        raise ValueError("Conflicting checkpoints at the same step. Attach the intended Stage 6 snapshot.")
    if kind != 2:
        best = unpack(best) / relative_run if kind == 0 else best
        assert sha(best / "checkpoint.pt") == digest
        assert checked_state(best, meta, fork_step, review_step)["step"] == step
        if run.exists():
            # Keep the older local run in place when an attached snapshot is further along.
            work = outputs / f"recovery-stage6-dense-extension-restored-{step}-{digest[:12]}"
    else:
        work = best.parents[len(relative_run.parts) - 1]
    run = work / relative_run
    manifest = work / "manifests/lego.json"
    manifest.parent.mkdir(parents=True, exist_ok=True)
    write_json(manifest, frozen)
else:
    best = None
    step = fork_step

if not run.exists():
    run.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=run.parent) as tmp:
        staging = Path(tmp) / "run"
        staging.mkdir()
        write_json(staging / "run.json", meta)
        if best is None:
            state = checked_state(parent_run, parent_meta, fork_step, fork_step)
            assert all(abs(g["lr"] - cfg["lr"]) < 1e-12 for g in state["optimizer"]["param_groups"])
            # The new fingerprint identifies this independent continuation. All trained tensors,
            # Adam slots, RNG states and the absolute step are copied from the verified parent.
            state["fingerprint"] = fingerprint(meta)
            torch.save(state, staging / "checkpoint.pt")
            del state
        else:
            for path in best.iterdir():
                if path.is_file() and path.name in {"checkpoint.pt", "training.jsonl", "health.json", "train-fit.json"}:
                    shutil.copyfile(path, staging / path.name)
            for preview in best.glob("train-*.png"):
                shutil.copyfile(preview, staging / preview.name)
        assert checked_state(staging, meta, fork_step, review_step)["step"] == step
        staging.rename(run)
print(f"Verified extension run {child_id} at {step:,}. Parent {parent_id} remains at {fork_step:,}.")

In [ ]:
def export_checkpoint(final=False):
    state = checked_state(run, meta, fork_step, review_step)
    step = state["step"]
    del state
    ckpt = run / "checkpoint.pt"
    receipt = dict(run_id=child_id, lineage=lineage, schedule=protocol_id, candidate_end=candidate_end,
                   checkpoint_step=step, checkpoint_sha256=sha(ckpt), data_hash=data_sha,
                   device_name=torch.cuda.get_device_name(device), torch_version=str(torch.__version__),
                   cuda=torch.version.cuda, train_fit_included=final)
    files = [manifest, run / "run.json", ckpt]
    health = run / "health.json"
    if health.exists() and read(health)["step"] == step:
        files.append(health)
    files.extend(p for p in run.glob("train-*.png") if int(p.stem.split("-")[1]) <= step)
    fit = run / "train-fit.json"
    if final:
        assert step == review_step and read(fit)["checkpoint_sha256"] == receipt["checkpoint_sha256"]
        files.append(fit)
    tag = "stage6-review" if final else f"extension-{step:06d}"
    target = outputs / f"lego-dense-vanilla-s42-{tag}.zip"
    receipt["artifact_sha256"] = {p.relative_to(work).as_posix(): sha(p) for p in files}
    pending = target.with_suffix(".tmp")
    with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for path in files:
            archive.write(path, path.relative_to(work).as_posix())
        log = run / "training.jsonl"
        if log.exists():
            # A training process can stop after logging an update but before committing its checkpoint.
            committed = []
            for line in log.read_text(encoding="utf-8").splitlines():
                try:
                    logged_step = json.loads(line)["step"]
                except (ValueError, KeyError):
                    continue
                if logged_step <= step:
                    committed.append(line)
            log_name = (relative_run / log.name).as_posix()
            payload = ("\n".join(committed) + "\n").encode()
            receipt["artifact_sha256"][log_name] = hashlib.sha256(payload).hexdigest()
            archive.writestr(log_name, payload)
        archive.writestr("stage6-review.json", json.dumps(receipt, allow_nan=False))
    with zipfile.ZipFile(pending) as archive:
        assert archive.testzip() is None
    os.replace(pending, target)
    print(f"Saved {target.name} at update {step:,}  SHA-256 {sha(target)}", flush=True)
    return target

def download(path):
    payload = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<a download="{path.name}" href="data:application/zip;base64,{payload}">Download {path.name}</a>'))

def segment(stop):
    start = checked_state(run, meta, fork_step, review_step)["step"]
    if start >= stop:
        print(f"Checkpoint already at {start:,}.")
        return
    try:
        train(run, manifest, "dense-vanilla", 42, device, cfg, resume=True, stop_at=stop, save_every=500)
    finally:
        download(export_checkpoint())

segment(55_000)

In [ ]:
segment(review_step)

In [ ]:
# These are the eight full-image and foreground PSNR pairs from the verified Stage 5 report.
reference = {
    "train/r_26.png": (29.752329336953704, 23.737326690749832),
    "train/r_86.png": (30.236970274341775, 25.60764675464261),
    "train/r_2.png": (30.184044346974872, 25.485065488326015),
    "train/r_55.png": (29.71697831843803, 24.08690094445952),
    "train/r_75.png": (30.879151792853893, 24.81590576560273),
    "train/r_93.png": (29.833731192622736, 25.453932524972824),
    "train/r_16.png": (30.230604948189157, 26.079534785850115),
    "train/r_73.png": (30.082020572835148, 25.68361576336619),
}
reference_sha = "b54f986c95e5dedc35476e43a57476fc909cd48dda8c43f8875fa37ea10d094d"
state = checked_state(run, meta, review_step, review_step)
checkpoint_sha = sha(run / "checkpoint.pt")
fit_path = run / "train-fit.json"
prior = read(fit_path) if fit_path.exists() else {}
if (prior.get("checkpoint_sha256"), prior.get("reference_sha256")) == (checkpoint_sha, reference_sha):
    fit = prior
else:
    model = make_model(cfg, device)
    model.load_state_dict(state["model"])
    model.eval()
    dataset = SceneDataset(manifest, "sparse", meta["downscale"])
    assert (dataset.width, dataset.height, dataset.white_background) == (400, 400, True)
    assert [dataset.frame_id(i) for i in range(len(dataset))] == list(reference)
    rows = []
    for index, (name, (baseline_psnr, baseline_fg)) in enumerate(reference.items()):
        started = time.monotonic()
        origins, directions, target = dataset.frame(index)
        rgb, _, opacity = render_maps(model, dataset, (origins, directions), False, device, 1024)
        mask = alpha_mask(dataset, index)
        score, fg = psnr(rgb, target), psnr(rgb[mask], target[mask])
        rows.append(dict(frame=name, psnr=score, foreground_psnr=fg, foreground_opacity=opacity[mask].mean().item(),
                         psnr_at_50000=baseline_psnr, foreground_psnr_at_50000=baseline_fg,
                         delta_psnr=score-baseline_psnr, delta_foreground_psnr=fg-baseline_fg,
                         seconds=time.monotonic()-started))
        print(f"[train {index+1}/8] {name}  {score:.3f} dB  change {score-baseline_psnr:+.3f} dB", flush=True)
    fit = dict(run_id=child_id, checkpoint_step=review_step, checkpoint_sha256=checkpoint_sha,
               reference_sha256=reference_sha, frames=rows,
               mean={key: statistics.mean(row[key] for row in rows) for key in (
                   "psnr", "foreground_psnr", "delta_psnr", "delta_foreground_psnr", "foreground_opacity")})
    write_json(fit_path, fit)
del state
print(json.dumps(fit["mean"], indent=2))
download(export_checkpoint(final=True))